In [1]:
import modeling_selfcheck_apiprompt
import torch
import spacy
import warnings
from openai import OpenAI
import os
warnings.filterwarnings("ignore")

selfcheck_apiprompt_solar_pro = modeling_selfcheck_apiprompt.SelfCheckAPIPrompt(client_type="openai", 
                                                                      client=OpenAI(api_key=os.environ["UPSTAGE_API_KEY"], 
                                                                                    base_url="https://api.upstage.ai/v1/solar"), 
                                                                        model="solar-pro")

Initiate OpenAI client ... model = solar-pro


In [2]:
import json

with open("data/dataset_v3.json", "r") as f:
    content = f.read()
dataset = json.loads(content)
print("The lenght of the dataset: {}".format(len(dataset)))
print(dataset[0].keys())

The lenght of the dataset: 238
dict_keys(['gpt3_text', 'wiki_bio_text', 'gpt3_sentences', 'annotation', 'wiki_bio_test_idx', 'gpt3_text_samples'])


In [3]:
import numpy as np

label_mapping = {
    'accurate': 0.0,
    'minor_inaccurate': 0.5,
    'major_inaccurate': 1.0,
}

human_label_detect_False   = {}
human_label_detect_True    = {}
human_label_detect_False_h = {}

for i_ in range(len(dataset)):
    dataset_i = dataset[i_]
    idx = dataset_i["wiki_bio_test_idx"]
    
    raw_label = np.array([label_mapping[x] for x in dataset_i['annotation']])
    human_label_detect_False[idx] = (raw_label > 0.499).astype(np.int32).tolist()
    human_label_detect_True[idx] = (raw_label < 0.499).astype(np.int32).tolist()
    average_score = np.mean(raw_label)
    if (average_score < 0.99):
        human_label_detect_False_h[idx] = (raw_label > 0.99).astype(np.int32).tolist()

In [6]:
import json

with open('data/selfcheck_scores_promptapi_solar_pro_backup.json', 'r') as file:
    scores_existing = json.load(file)

scores_existing

{'62464': [0.95, 0.95, 1.0, 1.0, 0.25, 0.7, 0.4, 0.95, 0.85],
 '49661': [0.95, 0.9, 1.0, 1.0, 1.0, 1.0, 1.0],
 '20483': [1.0, 0.9, 0.4, 0.55, 0.9, 0.9, 1.0, 0.05, 0.9],
 '71174': [1.0, 0.9, 1.0, 1.0, 0.75, 1.0, 1.0, 0.35, 1.0],
 '39945': [1.0, 0.35, 0.6, 0.95, 1.0, 0.4, 0.25, 0.0, 0.0, 0.35, 0.75],
 '26126': [0.7, 1.0, 0.7, 0.95, 0.95, 1.0, 0.35, 0.9, 0.75, 0.55, 0.9],
 '61454': [1.0, 1.0, 1.0, 1.0, 0.95, 1.0, 1.0, 0.95, 1.0],
 '37904': [0.65, 0.15, 0.0, 0.0, 0.15, 1.0, 0.05],
 '61460': [1.0, 0.55, 0.8, 0.55, 0.0, 0.85, 0.8, 1.0],
 '48151': [1.0, 0.9, 0.8, 1.0, 0.6, 0.65, 1.0, 1.0, 1.0],
 '71192': [0.1, 0.55, 0.3, 0.4, 0.05, 0.05],
 '17946': [0.15, 0.9, 0.35, 0.6, 0.85, 0.8, 0.75, 0.2, 0.55, 0.0, 0.5, 0.6],
 '21020': [0.85, 1.0, 0.9, 0.6, 0.7, 0.65, 0.9],
 '20508': [0.75, 0.05, 0.05, 0.0, 0.15, 0.05, 0.6, 0.7, 0.85, 0.85],
 '13854': [0.6, 0.8, 0.65, 0.95, 0.25, 1.0, 0.5, 0.55, 0.55],
 '1060': [1.0, 0.85, 0.5, 0.7, 0.85, 0.85, 0.85],
 '72743': [1.0, 1.0, 0.95, 0.95, 0.85, 0.85],
 '69672

In [8]:
print("Length of False:", len(human_label_detect_False))
print("Length of True:", len(human_label_detect_True)) 
print("Length of False_h:", len(human_label_detect_False_h))

Length of False: 238
Length of True: 238
Length of False_h: 206


In [17]:
from tqdm import tqdm

self_check_scores_solar_pro = {}

for i in tqdm(range(len(dataset))):
    x = dataset[i]
    idx = dataset[i]['wiki_bio_test_idx']

    if str(idx) in scores_existing:
        continue

    self_check_scores_solar_pro[idx] = selfcheck_apiprompt_solar_pro.predict(
        sentences = x['gpt3_sentences'],           
        sampled_passages = x['gpt3_text_samples'],
        verbose=True
    )

100%|██████████| 238/238 [1:06:14<00:00, 16.70s/it] 


In [18]:
selfcheck_scores_promptapi_solar_pro = {}

for idx in self_check_scores_solar_pro:
  scores = self_check_scores_solar_pro[idx]
  selfcheck_scores_promptapi_solar_pro[idx] = [score for score in scores]

with open("data/selfcheck_scores_promptapi_solar_pro.json", "w") as outfile:
    json.dump(selfcheck_scores_promptapi_solar_pro, outfile)